THÀNH VIÊN 1:  DATA PIPELINE & KHÁM PHÁ DỮ LIỆU (EDA & STYLIZED FACTS)

In [4]:
# Làm sạch dữ liệu (Data Cleaning)
# Cài cấu hình
import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

In [5]:
# Đọc cấu hình
BASE_DIR = Path(os.environ.get(
    "PROJECT_DIR",
    r"D:\Năm 3\Ứng dụng công nghệ trong TCNH\Project midterm",
))
INPUT_DIR = BASE_DIR / "Input"
OUTPUT_DIR = BASE_DIR / "Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
 
REQUIRED_COLS = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
FFILL_LIMIT = 10          # tối đa số ngày liên tiếp được điền giá (cổ phiếu tạm ngừng GD)
GAP_THRESHOLD_DAYS = 4    # khoảng cách giữa 2 phiên liên tiếp > 4 ngày lịch => ghi nhận (nghỉ Tết/lễ dài)
SUSPICIOUS_RET = 0.15     # biến động giá đóng cửa |r| > 15%/ngày => nghi ngờ chưa điều chỉnh chia tách
COMMON_SAMPLE = True      # True: chỉ giữ giai đoạn mà TẤT CẢ cổ phiếu cùng có dữ liệu
 
REPORT = []               # nhật ký xử lý, ghi ra file cuối cùng
 
 
def log(msg=""):
    print(msg)
    REPORT.append(str(msg))

In [6]:
# 1. ĐỌC DỮ LIỆU
def _map_col(col):
    """Chuẩn hoá tên cột (tiêu đề có thể kèm mô tả như 'Close price adjusted for splits.')."""
    c = str(col).strip().lower()
    if c.startswith("adj"):
        return "Adj Close"
    if c.startswith("date"):
        return "Date"
    if c.startswith("open"):
        return "Open"
    if c.startswith("high"):
        return "High"
    if c.startswith("low"):
        return "Low"
    if c.startswith("close"):
        return "Close"
    if c.startswith("volume"):
        return "Volume"
    return col
 
 
def load_stock(path: Path) -> pd.DataFrame:
    """Đọc 1 file Excel -> DataFrame index = Date, các cột số chuẩn hoá."""
    df = pd.read_excel(path, sheet_name=0)
    df.columns = [_map_col(c) for c in df.columns]
 
    missing = [c for c in ["Date"] + REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name}: thiếu cột {missing}. Cột hiện có: {list(df.columns)}")
 
    df = df[["Date"] + REQUIRED_COLS].copy()
 
    # Ngày: nếu là chuỗi thì ưu tiên định dạng dd/mm/yyyy
    if not pd.api.types.is_datetime64_any_dtype(df["Date"]):
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)
 
    for c in REQUIRED_COLS:
        if df[c].dtype == object:
            df[c] = df[c].astype(str).str.replace(",", "", regex=False)
        df[c] = pd.to_numeric(df[c], errors="coerce")
 
    n_bad_date = df["Date"].isna().sum()
    df = df.dropna(subset=["Date"])
    n_dup = df["Date"].duplicated().sum()
    df = (df.drop_duplicates(subset="Date", keep="last")
            .sort_values("Date")
            .set_index("Date"))
    df.index = df.index.normalize()
    df.index.name = "Date"
 
    log(f"  - Đọc {path.name}: {len(df)} dòng | {df.index.min().date()} -> {df.index.max().date()}"
        f" | ngày lỗi bỏ: {n_bad_date} | ngày trùng bỏ: {n_dup}")
    return df
 
 
def load_all() -> dict:
    files = sorted(p for p in INPUT_DIR.glob("*.xlsx") if not p.name.startswith("~$"))
    if not files:
        raise FileNotFoundError(f"Không tìm thấy file .xlsx trong {INPUT_DIR}")
    log("=" * 78)
    log("ĐỌC DỮ LIỆU")
    log("=" * 78)
    return {p.stem.upper(): load_stock(p) for p in files}

In [7]:
# BƯỚC 1 – XỬ LÝ GIÁ ĐÓNG CỬA ĐIỀU CHỈNH
def step1_adjusted_prices(data: dict) -> dict:
    """
    - Hệ số điều chỉnh  adj_factor = Adj Close / Close (phản ánh chia tách + cổ tức).
    - Điều chỉnh Open/High/Low theo cùng hệ số -> Adj Open / Adj High / Adj Low.
    - Kiểm tra: giá <= 0, High < Low, ngày có sự kiện doanh nghiệp, biến động bất thường.
    - Mọi phân tích lợi suất về sau dùng Adj Close (KHÔNG dùng Close).
    """
    log("")
    log("=" * 78)
    log("BƯỚC 1: XỬ LÝ GIÁ ĐÓNG CỬA ĐIỀU CHỈNH (ADJUSTED CLOSE)")
    log("=" * 78)
    out = {}
    for tk, df in data.items():
        df = df.copy()
 
        # Giá không hợp lệ -> NaN (sẽ xử lý ở bước 2)
        for c in ["Open", "High", "Low", "Close", "Adj Close"]:
            df.loc[df[c] <= 0, c] = np.nan
 
        df["Adj_Factor"] = df["Adj Close"] / df["Close"]
        df["Adj_Open"] = df["Open"] * df["Adj_Factor"]
        df["Adj_High"] = df["High"] * df["Adj_Factor"]
        df["Adj_Low"] = df["Low"] * df["Adj_Factor"]
 
        # Ngày hệ số điều chỉnh thay đổi => có chia tách / cổ tức / phát hành thêm
        chg = df["Adj_Factor"].pct_change().abs() > 1e-4
        n_events = int(chg.sum())
 
        # So sánh lợi suất theo Close và theo Adj Close
        r_close = df["Close"].pct_change()
        r_adj = df["Adj Close"].pct_change()
        n_diff = int(((r_close - r_adj).abs() > 1e-4).sum())
 
        # Biến động lớn trên Close mà hệ số không đổi => nghi ngờ chưa điều chỉnh
        suspicious = df.index[(r_adj.abs() > SUSPICIOUS_RET)]
        bad_hl = int((df["High"] < df["Low"]).sum())
 
        log(f"  [{tk}] hệ số điều chỉnh: min={df['Adj_Factor'].min():.4f}, "
            f"max={df['Adj_Factor'].max():.4f} | số ngày hệ số đổi (sự kiện): {n_events} "
            f"| ngày lợi suất Close ≠ Adj Close: {n_diff} | High<Low: {bad_hl}")
        if len(suspicious):
            ds = ", ".join(d.strftime("%Y-%m-%d") for d in suspicious[:8])
            log(f"      ⚠ {len(suspicious)} ngày |r_adj| > {SUSPICIOUS_RET:.0%} (cần kiểm tra thủ công): {ds}")
        out[tk] = df
    return out

In [ ]:
# BƯỚC 2 – XỬ LÝ NGÀY NGHỈ LỄ, TẾT VÀ NGÀY KHÔNG GIAO DỊCH
def step2_non_trading_days(data: dict) -> dict:
    """
    - Loại các dòng rơi vào Thứ 7/CN (nếu có), loại dòng thiếu Adj Close.
    - Gắn cờ Is_Traded: phiên có khối lượng > 0 (phiên khối lượng = 0 là tạm ngừng GD).
    - Thống kê các khoảng nghỉ dài (Tết, lễ 30/4–1/5, 2/9...) trong từng mã.
    """
    log("")
    log("=" * 78)
    log("BƯỚC 2: XỬ LÝ NGÀY NGHỈ LỄ, TẾT & NGÀY KHÔNG GIAO DỊCH")
    log("=" * 78)
    out = {}
    for tk, df in data.items():
        df = df.copy()
 
        wk = df.index.dayofweek >= 5
        n_wk = int(wk.sum())
        df = df[~wk]
 
        n_nan = int(df["Adj Close"].isna().sum())
        df = df.dropna(subset=["Adj Close"])
 
        df["Volume"] = df["Volume"].fillna(0)
        df["Is_Traded"] = df["Volume"] > 0
        n_zero_vol = int((~df["Is_Traded"]).sum())
 
        # Các khoảng nghỉ dài giữa 2 phiên liên tiếp
        gaps = df.index.to_series().diff().dt.days
        long_gaps = gaps[gaps > GAP_THRESHOLD_DAYS]
 
        log(f"  [{tk}] bỏ dòng cuối tuần: {n_wk} | bỏ dòng thiếu Adj Close: {n_nan} "
            f"| phiên Volume=0 (không GD): {n_zero_vol} | số kỳ nghỉ dài (>{GAP_THRESHOLD_DAYS} ngày): {len(long_gaps)}")
        for d, g in long_gaps.head(6).items():
            prev = d - pd.Timedelta(days=int(g))
            log(f"      nghỉ {int(g)} ngày: {prev.date()} -> {d.date()}")
        out[tk] = df
    return out

In [9]:
# BƯỚC 3 – CĂN CHỈNH LỊCH GIAO DỊCH (CALENDAR ALIGNMENT)
def step3_calendar_alignment(data: dict):
    """
    - Lịch chuẩn (master calendar) = hợp (union) các ngày làm việc có dữ liệu ở ÍT NHẤT 1 mã.
      Ngày không mã nào giao dịch (Tết, Quốc khánh...) tự động nằm ngoài lịch -> đúng lịch nghỉ thị trường.
    - Cắt về giai đoạn chung [max(ngày bắt đầu), min(ngày kết thúc)] nếu COMMON_SAMPLE=True.
    - Reindex từng mã theo lịch chuẩn:
        + Giá: forward-fill (tối đa FFILL_LIMIT ngày) -> lợi suất ngày đó = 0, có cờ Is_Traded = False.
        + Volume: điền 0.
    - Tạo panel dạng rộng: AdjClose, LogReturn, Volume, Is_Traded.
    """
    log("")
    log("=" * 78)
    log("BƯỚC 3: CĂN CHỈNH LỊCH GIAO DỊCH GIỮA CÁC CỔ PHIẾU")
    log("=" * 78)
 
    union = sorted(set().union(*[set(df.index) for df in data.values()]))
    master = pd.DatetimeIndex(union)
 
    if COMMON_SAMPLE:
        start = max(df.index.min() for df in data.values())
        end = min(df.index.max() for df in data.values())
        master = master[(master >= start) & (master <= end)]
        log(f"  Giai đoạn chung: {start.date()} -> {end.date()}")
    log(f"  Số phiên trong lịch chuẩn: {len(master)}")
 
    # Ngày làm việc (T2–T6) vắng khỏi lịch chuẩn = ngày nghỉ lễ/Tết của thị trường
    all_bdays = pd.bdate_range(master.min(), master.max())
    holidays = all_bdays.difference(master)
    log(f"  Số ngày làm việc nghỉ (lễ/Tết, không mã nào có dữ liệu): {len(holidays)}")
 
    aligned = {}
    for tk, df in data.items():
        n_before = len(df.loc[(df.index >= master.min()) & (df.index <= master.max())])
        a = df.reindex(master)
 
        a["Is_Traded"] = a["Is_Traded"].fillna(False).astype(bool)
        a["Volume"] = a["Volume"].fillna(0)
        for c in ["Open", "High", "Low", "Close", "Adj Close",
                  "Adj_Open", "Adj_High", "Adj_Low", "Adj_Factor"]:
            a[c] = a[c].ffill(limit=FFILL_LIMIT)
 
        n_filled = len(master) - n_before
        n_left_nan = int(a["Adj Close"].isna().sum())
        log(f"  [{tk}] có sẵn {n_before}/{len(master)} phiên | điền thêm {n_filled} phiên "
            f"| phiên không GD: {int((~a['Is_Traded']).sum())} | NaN còn lại: {n_left_nan}")
        aligned[tk] = a
 
    # Panel dạng rộng
    adj_close = pd.DataFrame({tk: a["Adj Close"] for tk, a in aligned.items()})
    volume = pd.DataFrame({tk: a["Volume"] for tk, a in aligned.items()})
    traded = pd.DataFrame({tk: a["Is_Traded"] for tk, a in aligned.items()})
    log_ret = np.log(adj_close).diff()
    simple_ret = adj_close.pct_change()
 
    panels = {
        "AdjClose": adj_close,
        "LogReturn": log_ret.iloc[1:],
        "SimpleReturn": simple_ret.iloc[1:],
        "Volume": volume,
        "Is_Traded": traded,
    }
    return aligned, panels, master, holidays

In [10]:
# LƯU KẾT QUẢ
def save_outputs(aligned, panels, holidays):
    # (a) Từng mã – dữ liệu sạch & đã căn lịch
    per_dir = OUTPUT_DIR / "clean_by_ticker"
    per_dir.mkdir(exist_ok=True)
    for tk, a in aligned.items():
        a.to_csv(per_dir / f"{tk}_clean.csv", encoding="utf-8-sig")
 
    # (b) Panel dạng rộng
    for name, p in panels.items():
        p.to_csv(OUTPUT_DIR / f"panel_{name}.csv", encoding="utf-8-sig")
 
    # (c) Dạng dài (long) – tiện cho groupby / statsmodels
    long = pd.concat(
        [a.assign(Ticker=tk).reset_index() for tk, a in aligned.items()],
        ignore_index=True,
    )
    long["LogReturn"] = long.groupby("Ticker")["Adj Close"].transform(lambda s: np.log(s).diff())
    long.to_csv(OUTPUT_DIR / "aligned_long.csv", index=False, encoding="utf-8-sig")
 
    # (d) Excel tổng hợp
    with pd.ExcelWriter(OUTPUT_DIR / "aligned_data.xlsx", engine="openpyxl") as xw:
        for name, p in panels.items():
            p.to_excel(xw, sheet_name=name)
        pd.DataFrame({"Holiday_Weekday": holidays}).to_excel(xw, sheet_name="Holidays", index=False)
 
    # (e) Danh sách ngày nghỉ
    pd.DataFrame({"Date": holidays, "Weekday": holidays.day_name()}).to_csv(
        OUTPUT_DIR / "market_holidays_detected.csv", index=False, encoding="utf-8-sig")
 
    # (f) Biểu đồ giá chuẩn hoá về 100
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        norm = panels["AdjClose"] / panels["AdjClose"].iloc[0] * 100
        ax = norm.plot(figsize=(11, 5), lw=1.1)
        ax.set_title("Adjusted Close chuẩn hoá (ngày đầu = 100)")
        ax.set_ylabel("Chỉ số"); ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "adj_close_normalized.png", dpi=150)
        plt.close()
    except Exception as e:  # matplotlib không bắt buộc
        log(f"  (Bỏ qua vẽ biểu đồ: {e})")

In [11]:
def final_checks(panels):
    log("")
    log("=" * 78)
    log("KIỂM TRA CUỐI")
    log("=" * 78)
    ret = panels["LogReturn"]
    log(f"  Kích thước panel lợi suất: {ret.shape[0]} ngày x {ret.shape[1]} mã")
    log(f"  Tổng số NaN trong panel lợi suất: {int(ret.isna().sum().sum())}")
    zero_share = (ret == 0).mean().round(4)
    log("  Tỷ lệ ngày lợi suất = 0 (do không GD/điền giá):")
    for tk, v in zero_share.items():
        log(f"     {tk}: {v:.2%}")
    log("\n  Thống kê mô tả log-return:")
    log(ret.describe().T[["count", "mean", "std", "min", "max"]].round(5).to_string())

In [14]:
raw = load_all()
print()
for tk, df in raw.items():
    print(f"{tk:8s} | {len(df):5d} dòng | {df.index.min()} -> {df.index.max()}")

ĐỌC DỮ LIỆU
  - Đọc FPT.xlsx: 1561 dòng | 2019-01-01 -> 2024-12-30 | ngày lỗi bỏ: 0 | ngày trùng bỏ: 18
  - Đọc HPG.xlsx: 1561 dòng | 2019-01-01 -> 2024-12-30 | ngày lỗi bỏ: 0 | ngày trùng bỏ: 8
  - Đọc MWG.xlsx: 1554 dòng | 2019-01-01 -> 2024-12-30 | ngày lỗi bỏ: 0 | ngày trùng bỏ: 8
  - Đọc VCB.xlsx: 1296 dòng | 2021-11-12 -> 2026-11-12 | ngày lỗi bỏ: 0 | ngày trùng bỏ: 6
  - Đọc VHM.xlsx: 1561 dòng | 2019-01-01 -> 2024-12-30 | ngày lỗi bỏ: 0 | ngày trùng bỏ: 4

FPT      |  1561 dòng | 2019-01-01 00:00:00 -> 2024-12-30 00:00:00
HPG      |  1561 dòng | 2019-01-01 00:00:00 -> 2024-12-30 00:00:00
MWG      |  1554 dòng | 2019-01-01 00:00:00 -> 2024-12-30 00:00:00
VCB      |  1296 dòng | 2021-11-12 00:00:00 -> 2026-11-12 00:00:00
VHM      |  1561 dòng | 2019-01-01 00:00:00 -> 2024-12-30 00:00:00
